# Subword Tokenizer

| **Tokenizer 방식** | **토큰 단위**                      | **vocab size** | **미등록 단어에 대한 가정**                                                                                  |
|---------------------|------------------------------------|----------------|-------------------------------------------------------------------------------------------------------------|
| **사전 기반**       | 알려진 단어/형태소의 결합           | unlimited       | - 알려진 단어/형태소의 결합이라고 가정<br>- 필요한 형태소 분석 가능<br>- 사전에 등록되지 않은 단어는 UNK 처리 |
| **sub-word**        | 알려진 글자 및 sub-word            | fixed           | - 알려진 sub-words로 분해<br>- 예: appear → app + ear<br>- 자주 등장하는 단어를 제대로 인식 가능<br>- UNK의 개수 최소화 |

### 네이버 영화리뷰 토크나이징

In [1]:
import urllib.request  # url을 받아 파일을 다운로드하는 모듈
import os              # 운영체제 조작 : 경로 / 폴더 생성 처리 모듈

# 파일 다운로드 함수 : 지정한 URL(origin)의 파일을 로컬 캐시에 저장하고 (없으면 다운로드) 경로 반환 함수
def get_file(filename, origin):
    cache_dir = os.path.expanduser('~/.torch/dataset')  # 사용자 홈(~) 아래 cache 디렉토리 경로 생성
    os.makedirs(cache_dir, exist_ok=True)  # 폴더가 없으면 생성(있으면 pass)
    filepath = os.path.join(cache_dir, filename)  # 파일의 전체 경로

    # 파일 없을시 다운로드
    if not os.path.exists(filepath):
        print(f"{origin} 파일 다운로드 중!")
        urllib.request.urlretrieve(origin, filepath)  # origin 파일 다운로드 후 filepath에 저장

    return filepath

In [2]:
ratings_train_path = get_file(
    'ratings_train.txt',
    'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt'
)

ratings_test_path = get_file(
    'ratings_test.txt',
    'https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt'
)

ratings_train_path, ratings_test_path

https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt 파일 다운로드 중!
https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt 파일 다운로드 중!


('C:\\Users\\uk/.torch/dataset\\ratings_train.txt',
 'C:\\Users\\uk/.torch/dataset\\ratings_test.txt')

In [3]:
import pandas as pd

ratings_train_df = pd.read_csv(ratings_train_path, sep="\t")  # 학습데이터를 df형식으로 로드(탭 구분)
ratings_test_df = pd.read_csv(ratings_test_path, sep="\t")

display(ratings_train_df)
display(ratings_test_df)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0
...,...,...,...
49995,4608761,오랜만에 평점 로긴했네ㅋㅋ 킹왕짱 쌈뽕한 영화를 만났습니다 강렬하게 육쾌함,1
49996,5308387,의지 박약들이나 하는거다 탈영은 일단 주인공 김대희 닮았고 이등병 찐따 OOOO,0
49997,9072549,그림도 좋고 완성도도 높았지만... 보는 내내 불안하게 만든다,0
49998,5802125,절대 봐서는 안 될 영화.. 재미도 없고 기분만 잡치고.. 한 세트장에서 다 해먹네,0


.txt 파일이어도 만약 현재처럼 id \t document \t label 처럼 표(테이블) 형식이면 read_csv로 구분자를 줘서 읽어낼 수 있다.

만약 id, document, label -> sep="," 로 읽어온다.

In [4]:
ratings_train_df.isnull().sum()

id          0
document    5
label       0
dtype: int64

In [5]:
ratings_test_df.isnull().sum()

id          0
document    3
label       0
dtype: int64

In [6]:
ratings_train_df = ratings_train_df.dropna(how='any')  # 하나라도 결측치가 있으면 해당 행 삭제 (기본값)
ratings_test_df = ratings_test_df.dropna(how='any')

ratings_train_df.shape, ratings_test_df.shape

((149995, 3), (49997, 3))

In [7]:
# 학습 리뷰 문장을 텍스트 파일에 저장
with open('naver_review.txt', 'w', encoding='utf-8') as f:
    for doc in ratings_train_df['document'].values:  # 학습 데이터 리뷰(document)들을 순회
        f.write(doc + '\n')  # 각 문장을 한 줄씩 파일에 기록

### SentencePieceTokenizer

In [12]:
%pip install sentencepiece

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 31.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sentencepiece as spt     # 서브워드 토크나이저

input = 'naver_review.txt'      # 학습에 사용할 파일
vocab_size = 10_000             # 만들 서브워드 사전 크기(토큰 개수)
model_prefix = 'naver_review'   # 저장될 모델/사전 파일 이름 접두어 (na~.model,na~.vocab)

# Sentencepiece는 내부적으로 CLI 명령줄을 받기 떄문에 만들어서 입력
cmd = f'--input={input} --model_prefix={model_prefix} --vocab_size={vocab_size}'
spt.SentencePieceTrainer.Train(cmd)

True

In [16]:
sp = spt.SentencePieceProcessor()
sp.Load(f"{model_prefix}.model")

for doc in ratings_train_df['document'].values[:3]:
    print(doc)
    print(sp.encode_as_pieces(doc))
    print(sp.encode_as_ids(doc))
    print()

아 더빙.. 진짜 짜증나네요 목소리
['▁아', '▁더빙', '..', '▁진짜', '▁짜증나', '네요', '▁목소리']
[63, 877, 5, 31, 2024, 69, 1714]

흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
['▁흠', '...', '포스터', '보고', '▁초딩', '영화', '줄', '....', '오', '버', '연기', '조차', '▁가볍지', '▁않', '구나']
[1633, 8, 4932, 157, 1281, 33, 269, 62, 171, 577, 419, 1231, 7426, 756, 448]

너무재밓었다그래서보는것을추천한다
['▁너무', '재', '밓', '었다', '그래서', '보는', '것을', '추천', '한다']
[22, 380, 9759, 427, 3788, 519, 2540, 1958, 331]

